# 06 - Avaliacao e comparacao final

**Objetivo:** consolidar metricas, rankings, matriz de confusao e tabelas finais para a secao de Resultados.

Este notebook nao inventa metricas: ele consolida apenas arquivos gerados pelos notebooks anteriores.

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from tcc_ecg.config import load_config
from tcc_ecg.evaluation import save_classification_report_tables
from tcc_ecg.paths import resolve_project_path
from tcc_ecg.plots import plot_confusion_matrix, plot_metrics_comparison
from tcc_ecg.utils import save_table

config = load_config()
tables_dir = resolve_project_path(config['outputs']['tables_dir'], config['project_root'])
figures_dir = resolve_project_path(config['outputs']['figures_dir'], config['project_root'])

frequency = int(config['data']['signal_frequency'])
metric_files = [
    tables_dir / f'model_metrics_{frequency}hz.csv',
    tables_dir / f'deep_learning_baseline_metrics_{frequency}hz.csv',
    tables_dir / f'deep_learning_metrics_{frequency}hz.csv',
    tables_dir / f'deep_learning_resnet1d_metrics_{frequency}hz.csv',
    tables_dir / 'model_metrics.csv',
    tables_dir / 'deep_learning_baseline_metrics.csv',
    tables_dir / 'deep_learning_metrics.csv',
    tables_dir / 'deep_learning_resnet1d_metrics.csv',
]
metric_frames = [pd.read_csv(path) for path in metric_files if path.exists()]
if not metric_frames:
    raise FileNotFoundError('Execute os notebooks 04 e/ou 05 antes da avaliacao final.')
metrics = pd.concat(metric_frames, ignore_index=True)
if 'signal_frequency' not in metrics.columns:
    metrics = metrics.assign(signal_frequency=frequency)
metrics = metrics.drop_duplicates(subset=['model', 'split', 'signal_frequency'], keep='first')

final_comparison = metrics.loc[metrics['split'].eq('test')].sort_values('f1_macro', ascending=False)
if 'signal_frequency' not in final_comparison.columns:
    final_comparison = final_comparison.assign(signal_frequency=frequency)
save_table(final_comparison, tables_dir / 'final_model_comparison.csv', tables_dir / 'final_model_comparison.tex')
save_table(final_comparison, tables_dir / f'final_model_comparison_{frequency}hz.csv', tables_dir / f'final_model_comparison_{frequency}hz.tex')
plot_metrics_comparison(metrics, figures_dir / 'fig_metrics_comparison.png')
display(final_comparison)

,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro,f1_weighted,model,split,smote,signal_frequency
7,0.692121,0.522279,0.618911,0.522279,0.556135,0.674490,lightgbm_without_smote,test,False,500
17,0.680606,0.533515,0.583156,0.533515,0.553702,0.668058,lightgbm_with_smote,test,True,500
19,0.641818,0.568425,0.523141,0.568425,0.538179,0.646572,catboost_with_smote,test,True,500
15,0.632121,0.550433,0.514603,0.550433,0.527852,0.633163,random_forest_with_smote,test,True,500
9,0.621212,0.577807,0.507821,0.577807,0.526707,0.633126,catboost_without_smote,test,False,500
1,0.593333,0.586476,0.492293,0.586476,0.509946,0.615582,logistic_regression_without_smote,test,False,500
11,0.583030,0.572623,0.482067,0.572623,0.499611,0.605228,logistic_regression_with_smote,test,True,500
13,0.550909,0.575225,0.474778,0.575225,0.483242,0.579418,svm_with_smote,test,True,500
3,0.600606,0.505072,0.465305,0.505072,0.479385,0.603371,svm_without_smote,test,False,500
5,0.659394,0.426011,0.648196,0.426011,0.469691,0.606878,random_forest_without_smote,test,False,500


In [2]:
best_model_name = final_comparison.iloc[0]['model']
prediction_path = tables_dir / f'test_predictions_{best_model_name}.csv'
if not prediction_path.exists():
    print(f'Predicoes do melhor modelo nao encontradas: {prediction_path}')
else:
    predictions = pd.read_csv(prediction_path)
    class_names = config['labels']['superclasses']
    plot_confusion_matrix(
        predictions['target_id'], predictions['y_pred'], class_names,
        figures_dir / 'fig_best_model_confusion_matrix.png',
        title=f'Matriz de confusao - {best_model_name}',
    )
    plot_confusion_matrix(
        predictions['target_id'], predictions['y_pred'], class_names,
        figures_dir / 'fig_best_model_confusion_matrix_normalized.png',
        normalize='true',
        title=f'Matriz de confusao normalizada - {best_model_name}',
    )
    report = save_classification_report_tables(
        predictions['target_id'], predictions['y_pred'], class_names, config,
        stem='best_model_classification_report',
    )
    display(report)

,class,precision,recall,f1-score,support
0,NORM,0.753006,0.892544,0.816859,912.000000
1,MI,0.552147,0.351562,0.429594,256.000000
2,STTC,0.513393,0.475207,0.493562,242.000000
3,CD,0.695364,0.570652,0.626866,184.000000
4,HYP,0.580645,0.321429,0.413793,56.000000
5,accuracy,0.692121,0.692121,0.692121,0.692121
6,macro avg,0.618911,0.522279,0.556135,1650.000000
7,weighted avg,0.674422,0.692121,0.674490,1650.000000
